In [2]:
import math
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ==========================================
# CALIBRATION CONSTANTS
# ==========================================
SYSTEM_GAIN_FACTOR = 1.05e-8
OSM_SLOPE = 15.5
OSM_INTERCEPT = 280.0

# ==========================================
# CLINICAL LOGIC
# ==========================================
def calculate_osmolarity(real, imag, current_temp):
    """Converts raw electrical impedance into clinical osmolarity."""
    if real == 0 and imag == 0: return 0, 0

    # Calculate Magnitude and Raw Impedance
    magnitude = math.sqrt((real**2) + (imag**2))
    raw_impedance = 1.0 / (SYSTEM_GAIN_FACTOR * magnitude)

    # Thermodynamic Compensation (Tears fluctuate by ~2% per degree C)
    temp_coefficient = 0.02
    comp_impedance = raw_impedance / (1 + temp_coefficient * (current_temp - 25.0))

    # Convert to Conductivity and Map to Osmolarity
    conductivity = 1.0 / comp_impedance
    osmolarity = (conductivity * OSM_SLOPE) + OSM_INTERCEPT

    return comp_impedance, osmolarity

def generate_diagnosis(osmolarity):
    """Categorizes the severity and provides preventive care suggestions."""
    if osmolarity < 280:
        return ("Atypical (Hypoosmolar)", "yellow", "Result is unusually low. Ensure sample was not diluted and recalibrate sensors.")
    elif 280 <= osmolarity <= 295:
        return ("Normal (Healthy Tear Film)", "green", "Maintain current eye care routine. Remember the 20-20-20 rule during screen time (every 20 mins, look 20 ft away for 20 secs).")
    elif 296 <= osmolarity <= 316:
        return ("Mild to Moderate Dry Eye", "orange", "Use preservative-free artificial tears 2-4 times daily. Consider a bedside humidifier and reduce direct AC exposure.")
    else:
        return ("Severe Dry Eye Disease", "red", "Immediate clinical intervention recommended. Consult an ophthalmologist for prescription anti-inflammatory drops (e.g., Restasis) or punctal plugs.")

# ==========================================
# INTERACTIVE UI BUILDER
# ==========================================
# Input Sliders
temp_slider = widgets.FloatSlider(value=25.0, min=20.0, max=40.0, step=0.1, description='Temp (°C):')
real_slider = widgets.IntSlider(value=15000, min=1000, max=30000, step=100, description='Real (R):')
imag_slider = widgets.IntSlider(value=-5000, min=-30000, max=-1000, step=100, description='Imag (I):')
analyze_button = widgets.Button(description="Analyze Sample", button_style='info', icon='stethoscope')

# Output Display Area
out = widgets.Output()

def on_analyze_clicked(b):
    with out:
        clear_output()
        # Fetch slider values
        t = temp_slider.value
        r = real_slider.value
        i = imag_slider.value

        # Run calculations
        impedance, osmolarity = calculate_osmolarity(r, i, t)
        diagnosis, color, care = generate_diagnosis(osmolarity)

        # Render beautiful HTML output
        html_output = f"""
        <div style="border: 2px solid {color}; padding: 20px; border-radius: 10px; background-color: #f0f0f0;">
            <h3><b>Osmolarity Analysis Result</b></h3>
            <p><b>Input Parameters:</b> Temp={t}°C, Real={r}, Imag={i}</p>
            <p><b>Compensated Impedance:</b> {impedance:.2f} kΩ</p>
            <p><b>Calculated Osmolarity:</b> {osmolarity:.2f} mOsm/L</p>
            <p><b>Diagnosis:</b> <span style="color: {color};"><b>{diagnosis}</b></span></p>
            <p><b>Preventive Care:</b> {care}</p>
        </div>
        """
        display(HTML(html_output))

# Link button to function
analyze_button.on_click(on_analyze_clicked)

# Arrange and display widgets
display(
    widgets.VBox([
        temp_slider,
        real_slider,
        imag_slider,
        analyze_button,
        out
    ])
)
